In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
PROJECT_DIR = '/content/drive/MyDrive/fake-job-detector'
%cd $PROJECT_DIR

/content/drive/MyDrive/fake-job-detector


In [18]:
import sklearn
print(sklearn.__version__)
!python --version

1.6.1
Python 3.13.16


In [19]:
import pandas as pd


In [20]:
from src.config import DATA_PATH, TEXT_COLS, TARGET_COL

In [21]:
import pandas as pd
from src.config import TARGET_COL

df_clean = pd.read_csv('data/df_clean.csv')
print(df_clean.shape)
print(df_clean.columns.tolist())

(17880, 3)
['combined_text', 'clean_text', 'fraudulent']


In [22]:
from sklearn.model_selection import train_test_split
from src.config import TEST_SIZE, RANDOM_STATE

In [23]:
X = df_clean['clean_text']
y = df_clean[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(X_train.shape, X_test.shape)
print(y_train.mean())

(14304,) (3576,)
0.04844798657718121


In [24]:
from sklearn.pipeline import Pipeline

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [26]:
from sklearn.linear_model import LogisticRegression

In [27]:
pipe_baseline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        max_features=10000,
        ngram_range=(1, 2),
        min_df=3,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])

In [28]:
pipe_baseline.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=10000, min_df=3,
                                 ngram_range=(1, 2), stop_words='english',
                                 sublinear_tf=True)),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [29]:
from sklearn.metrics import classification_report, confusion_matrix

In [30]:
y_pred_baseline = pipe_baseline.predict(X_test)
print(classification_report(y_test, y_pred_baseline, digits=3))
print(confusion_matrix(y_test, y_pred_baseline))

              precision    recall  f1-score   support

           0      0.995     0.984     0.989      3403
           1      0.739     0.902     0.812       173

    accuracy                          0.980      3576
   macro avg      0.867     0.943     0.901      3576
weighted avg      0.983     0.980     0.981      3576

[[3348   55]
 [  17  156]]
